# From Zero to Claude: A Contract-Analysis Story

Learn how to analyze real legal contracts with Claude — from how LLMs work, to the two ways of calling Claude in production: the **Messages API** and the **Claude Agent SDK**.

| # | Chapter | You will learn |
|---|---|---|
| 1 | **Setup** | Install tools, load two contracts |
| 2 | **How LLMs Behave** | Tokens, context window, sampling, non-determinism, evaluation |
| 3 | **Model & Reasoning** | Pick a model, control thinking |
| 4 | **Prompting** | Zero-, one- and few-shot prompts |
| 5 | **Messages API** | Sync, streaming, async, batch |
| 6 | **Agent SDK vs. Messages API** | Same task, two interfaces |

### Prerequisites
Download both contracts (you'll upload them in Chapter 1):
- 📄 [AWS1.pdf](https://drive.google.com/file/d/1XSe2pSsGN1ssAbif92rvb80AnHb_Ni0F/view?usp=sharing) — short, ~12 pages
- 📚 [PROFRAC HOLDINGS, LLC credit agreement.pdf](https://drive.google.com/file/d/1UyOxeaEQsK5TFxXHI63PshoKmj0yjTmW/view?usp=sharing) — long, ~216 pages

## Chapter 1 — Setup

Install the libraries we need:

| Package | Used for |
|---|---|
| `anthropic` | Calling Claude (Messages API) |
| `PyMuPDF` | Extracting text from PDFs |
| `tiktoken` | Visualizing tokenization |
| `pyboxen` | Printing responses in readable boxes |

The Claude Agent SDK is installed later, in Chapter 6, when we need it.

In [ ]:
!pip install -q -U anthropic PyMuPDF tiktoken pyboxen

### 🔑 API key
Paste your **Anthropic** API key when prompted (it is not stored in the notebook). We also save it as `ANTHROPIC_API_KEY`, which the SDKs read automatically — including the Agent SDK in Chapter 6.

In [ ]:
import os, base64, asyncio
from getpass import getpass

import anthropic
import fitz  # PyMuPDF
import tiktoken
from pyboxen import boxen

os.environ["ANTHROPIC_API_KEY"] = getpass("Anthropic API key: ")
client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY


def show(title, text, color="cyan"):
    """Print text inside a titled, colored box."""
    print(boxen(str(text), title=title, color=color, padding=1))

### 📤 Upload the contracts
Run the next cell and select **both** PDFs: `AWS1.pdf` and `PROFRAC HOLDINGS, LLC credit agreement.pdf`.

In [ ]:
from google.colab import files
uploaded = files.upload()

Now extract the text, page by page. `contract_text` (the short AWS contract) is what we'll use for most of the notebook.

In [ ]:
SHORT_PDF = "/content/AWS1.pdf"
LONG_PDF = "/content/PROFRAC HOLDINGS, LLC credit agreement.pdf"


def extract_text(pdf_path):
    """Return the full text of a PDF, with one string per page."""
    with fitz.open(pdf_path) as doc:
        return [page.get_text() for page in doc]


short_pages = extract_text(SHORT_PDF)
long_pages = extract_text(LONG_PDF)

contract_text = "\n".join(short_pages)  # short contract as one string
long_text = "\n".join(long_pages)       # long contract as one string

print(f"Short contract: {len(short_pages)} pages | Long contract: {len(long_pages)} pages")

## Chapter 2 — How LLMs Behave

Imagine an assistant where a user uploads a contract and asks:
> *"What is the termination notice period?"*

Before adding RAG or agents, we need to understand what happens when that request reaches an LLM. Five ideas:

1. **Tokens** — how text is processed
2. **Context window** — how much fits in one request
3. **Sampling** — how the next word is chosen
4. **Non-determinism** — why answers vary
5. **Evaluation** — how to test such a system

### 1. 🧩 Tokens
LLMs don't read characters or words — they read **tokens**, small chunks of text (a word, part of a word, or punctuation) mapped to integer IDs.

Below we tokenize the first page of the contract. We use OpenAI's `cl100k_base` tokenizer purely to *illustrate* the idea; Claude has its own tokenizer, but the concept is identical.

In [ ]:
enc = tiktoken.get_encoding("cl100k_base")
page = short_pages[0]
ids = enc.encode(page)

show("Original text (first 200 chars)", page[:200].replace("\n", " ") + "...", "yellow")
show("First 20 tokens, split with |", "|".join(enc.decode([t]) for t in ids[:20]), "magenta")
print("Token IDs:", ids[:20])
print(f"\nCharacters: {len(page):,} | Tokens: {len(ids):,} | ≈ {len(page) / len(ids):.1f} characters per token")

**Takeaway:** ~4 characters ≈ 1 token in English. Tokens — not characters or pages — are what you pay for and what limits a request.

### 2. 🪟 Context Window
The **context window** is the maximum number of tokens the model can handle in one request — its *working memory*. The contract, your instructions **and** the model's answer all have to fit.

Let's measure both contracts with Claude's token counter (free to call):

In [ ]:
HAIKU = "claude-haiku-4-5"
SONNET = "claude-sonnet-4-6"

for name, text in [("Short contract", contract_text), ("Long contract", long_text)]:
    n = client.messages.count_tokens(
        model=HAIKU, messages=[{"role": "user", "content": text}]
    ).input_tokens
    print(f"{name}: {n:,} tokens")

Separately from tokens, the API also limits **PDF uploads** (max 100 pages per request). Let's send each PDF directly to Claude as a document and see what happens:

In [ ]:
def pdf_block(path):
    """Wrap a PDF file as a base64 'document' block that Claude can read."""
    with open(path, "rb") as f:
        data = base64.standard_b64encode(f.read()).decode()
    return {"type": "document", "source": {"type": "base64", "media_type": "application/pdf", "data": data}}


for name, path in [("Short (≈12 pages)", SHORT_PDF), ("Long (≈216 pages)", LONG_PDF)]:
    try:
        r = client.messages.create(
            model=HAIKU,
            max_tokens=1000,
            messages=[{"role": "user", "content": [
                pdf_block(path),
                {"type": "text", "text": "Summarize the key terms of this contract."},
            ]}],
        )
        show(f"✅ {name}", r.content[0].text, "green")
    except anthropic.APIError as e:
        show(f"❌ {name} — rejected", e.message, "red")

**Takeaway:** Every request has limits (tokens *and* PDF pages). Long documents don't always fit, which is why techniques like **chunking and RAG** exist. For the rest of this notebook we send the contract as plain text.

### 3. 🎲 Sampling & Temperature
An LLM generates text **one token at a time**. At each step it assigns a probability to every possible next token, then **samples** one:

```text
Next token after "Notice period is ..."
"30"     60%     "thirty" 20%     "90" 10%     other 10%
```

**`temperature`** (0–1) controls that sampling:
- **Low (0)** → almost always picks the most likely token → consistent
- **High (1)** → more willing to pick less likely tokens → varied

We'll ask the same question 3 times at each temperature and compare:

In [ ]:
question = (
    "In one sentence, describe how either party can terminate this contract.\n\n" + contract_text
)

for temp in (0, 1):
    for run in (1, 2, 3):
        r = client.messages.create(
            model=HAIKU,
            max_tokens=150,
            extra_body={"temperature": temp},
            messages=[{"role": "user", "content": question}],
        )
        show(f"temperature={temp} · run {run}", r.content[0].text, "blue" if temp == 0 else "red")

### 4. 🔀 Non-determinism
At temperature 1 the same input produces different wording. Even at temperature 0, outputs are not guaranteed to be identical.

```text
Run 1 → "The notice period is 30 days."
Run 2 → "Either party must give 30 days' notice."
```
Different words, **same meaning**. So we can't test LLM apps by checking for an exact string.

### 5. ✅ Evaluation
Traditional software tests use **exact match**: `expected == actual`. For LLMs, many different answers can be correct, so we judge the answer on:

| Criterion | Question |
|---|---|
| **Correctness** | Is the extracted term right? |
| **Relevance** | Does it answer the question asked? |
| **Completeness** | Is anything important missing? |
| **Grounding** | Is it supported by the contract text? |

> Don't ask *"Did it produce exactly this sentence?"* — ask *"Is the answer correct?"*

## Chapter 3 — Model & Reasoning

Not every task needs the same intelligence:
- Extract a date → **simple**
- Summarize a clause → **moderate**
- Judge whether a termination clause is risky → **complex**

### Choosing a model
There's no "best" model — only trade-offs between **capability, latency and cost**:

| Model | Capability | Latency | Cost | Good for |
|---|---|---|---|---|
| **Haiku** | Medium | 🟢 Low | 🟢 Low | Simple extraction, high volume |
| **Sonnet** | High | 🟡 Medium | 🟡 Medium | General analysis (our default) |
| **Opus** | Very high | 🔴 Higher | 🔴 Higher | Hardest reasoning |

### Reasoning (thinking)
Reasoning means giving the model tokens to *think* before it answers. More thinking → better on hard tasks, but slower and costlier.

| Mode | Code | Behavior |
|---|---|---|
| **Off** | `thinking={"type": "disabled"}` | Answers directly |
| **Fixed budget** | `thinking={"type": "enabled", "budget_tokens": 5000}` | Thinks up to a budget you set |
| **Adaptive** ✨ | `thinking={"type": "adaptive"}` | Claude decides *if and how much* to think |

`output_config={"effort": "high"}` sets how hard Claude tries. Responses contain separate **thinking** and **text** blocks, so we can see both:

> Defaults differ by model: Sonnet 4.6 has thinking off; Sonnet 5 and Opus 5 use adaptive thinking.

In [ ]:
response = client.messages.create(
    model=SONNET,
    max_tokens=4000,
    thinking={"type": "adaptive"},
    output_config={"effort": "high"},
    messages=[{"role": "user", "content": f"""
You are a contract risk analyst. For the AWS Customer Agreement below:
1. What termination rights does each party have?
2. What notice periods apply?
3. What happens to the customer's access and content after termination?
4. Is there significant risk for the customer?
5. Give the reviewer a concise recommendation.
Cite section numbers.

Contract:
{contract_text}
"""}],
)

for block in response.content:
    if block.type == "thinking":
        show("🧠 Thinking", block.thinking, "purple")
    elif block.type == "text":
        show("💬 Answer", block.text, "green")

## Chapter 4 — Prompting: Zero-, One- & Few-Shot

**Prompting** = giving the model instructions, context and (optionally) examples. The number of examples is the difference:

| Style | Examples | Use when |
|---|:-:|---|
| **Zero-shot** | 0 | The task is simple and self-explanatory |
| **One-shot** | 1 | You want to show the exact output format |
| **Few-shot** | 2–4 | There are different cases or edge cases (e.g. missing values) |

We'll extract the same key terms three ways. First, a small helper to avoid repeating the API call:

In [ ]:
def ask(prompt, model=HAIKU, max_tokens=1000, **kwargs):
    """Send one user prompt and return Claude's text answer."""
    r = client.messages.create(
        model=model, max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}], **kwargs,
    )
    return r.content[0].text

### 🎯 Zero-shot — instructions only

In [ ]:
zero_shot = f"""
Extract these key terms from the contract: parties, effective date, expiration date,
contract value, payment terms, termination terms, governing law.

Contract:
{contract_text}
"""

show("🎯 Zero-shot", ask(zero_shot))

### 🥇 One-shot — one example fixes the format

In [ ]:
one_shot = f"""
Extract the key terms as JSON, following the example. Return only JSON.

Example
Contract: "ABC Corp and XYZ Ltd agree to a 12-month term starting January 1, 2025, for $50,000,
payable net 30. Either party may terminate with 30 days' notice. Governed by New York law."
Output:
{{"parties": ["ABC Corp", "XYZ Ltd"], "effective_date": "2025-01-01", "expiration_date": "2025-12-31",
  "contract_value": 50000, "payment_terms": "Net 30", "termination_terms": "30 days' notice by either party",
  "governing_law": "New York"}}

Contract:
{contract_text}
Output:
"""

show("🥇 One-shot", ask(one_shot))

### 🏅 Few-shot — several examples cover edge cases (here: missing values → `null`)

In [ ]:
few_shot = f"""
Extract the key terms as JSON. Use null when a term is not stated. Return only JSON.

Example 1
Contract: "ABC Corp and XYZ Ltd agree to a 12-month term from January 1, 2025 for $50,000, net 30."
Output: {{"parties": ["ABC Corp", "XYZ Ltd"], "effective_date": "2025-01-01", "expiration_date": "2025-12-31",
          "contract_value": 50000, "payment_terms": "Net 30", "termination_terms": null, "governing_law": null}}

Example 2
Contract: "This agreement is effective March 3, 2024 between Acme Inc and Beta LLC. Either party may
terminate on 60 days' notice. Delaware law governs."
Output: {{"parties": ["Acme Inc", "Beta LLC"], "effective_date": "2024-03-03", "expiration_date": null,
          "contract_value": null, "payment_terms": null, "termination_terms": "60 days' notice by either party",
          "governing_law": "Delaware"}}

Example 3
Contract: "Services are provided by Orion Corp to Nova Ltd. Fees are USD 8,000 per month."
Output: {{"parties": ["Orion Corp", "Nova Ltd"], "effective_date": null, "expiration_date": null,
          "contract_value": null, "payment_terms": "USD 8,000 per month", "termination_terms": null, "governing_law": null}}

Contract:
{contract_text}
Output:
"""

show("🏅 Few-shot", ask(few_shot))

### What did we observe?
- **Zero-shot** — correct content, but the format is up to the model.
- **One-shot** — the output now follows our JSON structure.
- **Few-shot** — the model also handles *missing* values consistently (`null`).

> More examples ≠ always better. Ask: **"What is the smallest amount of guidance that gives the quality we need?"** If prompts keep growing, also consider a stronger model, more reasoning effort, or tighter output constraints.

## Chapter 5 — Talking to Claude: The Messages API

Every call so far used the **Messages API**: you send messages, you get a response. What changes is *how* you wait for it:

| Pattern | Idea | Use when |
|---|---|---|
| **Synchronous** | Send → wait → full response | Simple scripts |
| **Streaming** | Receive the answer piece by piece | Chat UIs that should feel instant |
| **Async** | Send many requests concurrently | You need several answers fast |
| **Batch** | Submit a background job, collect results later | Many independent jobs, e.g. evals |

```text
Sync:   A ──wait──▶ resp A ; B ──wait──▶ resp B        (one after another)
Async:  A ─┐
        B ─┼──▶ Claude ──▶ responses                   (all at once)
        C ─┘
```

### 1️⃣ Synchronous — blocks until the full answer arrives

In [ ]:
prompt = f"""
Identify the parties, effective date, expiration date, contract value and termination terms.

Contract:
{contract_text}
"""

response = client.messages.create(
    model=SONNET,
    max_tokens=1000,
    messages=[{"role": "user", "content": prompt}],
)
show("1️⃣ Synchronous", response.content[0].text)

### 2️⃣ Streaming — text arrives as it is generated
`client.messages.stream(...)` yields chunks through `stream.text_stream`. Watch the text appear live:

In [ ]:
with client.messages.stream(
    model=SONNET,
    max_tokens=1000,
    messages=[{"role": "user", "content": "Summarize the termination rights and notice periods.\n\n" + contract_text}],
) as stream:
    for chunk in stream.text_stream:
        print(chunk, end="", flush=True)

### 3️⃣ Async — don't block while waiting
`AsyncAnthropic` lets Python do other work while a request is in flight. Define the function with `async def` and wait for the result with `await`. (Colab allows `await` directly in a cell.)

In [ ]:
async_client = anthropic.AsyncAnthropic()  # reads ANTHROPIC_API_KEY


async def ask_async(prompt, max_tokens=500):
    r = await async_client.messages.create(
        model=SONNET, max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}],
    )
    return r.content[0].text


show("3️⃣ Async", await ask_async("Summarize the termination rights and risks.\n\n" + contract_text))

### 4️⃣ Many async calls in parallel
`asyncio.gather` runs all five requests **at the same time**, so total time ≈ the slowest request, not the sum.

In [ ]:
topics = ["contract parties", "contract value", "effective date", "termination terms", "governing law"]

answers = await asyncio.gather(
    *(ask_async(f"Identify the {t}.\n\nContract:\n{contract_text}") for t in topics)
)

for topic, answer in zip(topics, answers):
    show(f"4️⃣ {topic}", answer)

### 5️⃣ Message Batches — background jobs for evals
Say we want the **same 3 questions** answered for **both contracts** → 2 × 3 = **6 independent requests**. Instead of sending them one by one, submit them as one **batch**, then check back for the results.

```text
2 contracts × 3 questions ──▶ Batch ──▶ poll status ──▶ fetch results
```

**Step 1 — build and submit the 6 requests**

In [ ]:
questions = [
    "What is the total contract value?",
    "What are the payment terms?",
    "What are the termination conditions?",
]

batch_requests = [
    {
        "custom_id": f"{name}_q{i}",  # lets us match each result to its request
        "params": {
            "model": SONNET,
            "max_tokens": 500,
            "messages": [{"role": "user", "content":
                f"Contract:\n{text}\n\nQuestion: {q}\nAnswer only using information from the contract."}],
        },
    }
    for name, text in [("12_page", contract_text), ("216_page", long_text)]
    for i, q in enumerate(questions, 1)
]

batch = client.messages.batches.create(requests=batch_requests)
print(f"Submitted {len(batch_requests)} requests · batch id: {batch.id}")

**Step 2 — wait for completion, then read the results** (batches run in the background and can take a few minutes)

In [ ]:
import time

while (status := client.messages.batches.retrieve(batch.id)).processing_status != "ended":
    print("Batch status:", status.processing_status)
    time.sleep(5)

for item in client.messages.batches.results(batch.id):
    if item.result.type == "succeeded":
        show(item.custom_id, item.result.message.content[0].text, "green")
    else:
        show(item.custom_id, f"Request {item.result.type}", "red")

### Sync vs. async vs. batch

| | Question you're asking | Result arrives |
|---|---|---|
| **Sync** | "I need this answer now." | Immediately |
| **Async** | "I need several answers, fast." | Concurrently, in seconds |
| **Batch** | "I have many independent jobs and can wait." | Later, in the background (and at a lower price) |

## Chapter 6 — Claude Agent SDK vs. the Messages API

So far **we** wrote the prompt, called `client.messages.create(...)`, and got one response back. That's the **Messages API**: low-level, request in → response out.

The **Claude Agent SDK** works differently: you give Claude a *task*, and it can loop over multiple turns, use tools (files, bash, search, your own tools), and stream back a sequence of typed messages.

| | Messages API | Claude Agent SDK |
|---|---|---|
| **Call** | `client.messages.create()` | `query()` |
| **Shape** | 1 request → 1 response | 1 task → stream of messages, many turns |
| **Tools** | You run the tool loop yourself | Built-in loop and tools |
| **Best for** | Single-shot generation and extraction | Multi-step, tool-using workflows |

### Install
The SDK runs on the Claude Code CLI (needs Node.js, already in Colab). It uses the `ANTHROPIC_API_KEY` we set in Chapter 1.

In [ ]:
!npm install -g @anthropic-ai/claude-code
!pip install -q -U claude-agent-sdk

### Same task, two interfaces

**A) Messages API** — one call, one response object:

In [ ]:
task_prompt = f"""
Analyze this contract and identify: 1. parties, 2. effective date, 3. contract value, 4. termination terms.

Contract:
{contract_text}
"""

response = client.messages.create(
    model=SONNET, max_tokens=1000,
    messages=[{"role": "user", "content": task_prompt}],
)
show("📨 Messages API", response.content[0].text)  # the answer is right here

**B) Claude Agent SDK** — `query()` is async and yields a **stream** of messages. Claude's answer is inside `AssistantMessage` → `TextBlock`, so we have to pick it out of the stream. (The same loop would carry tool calls if Claude decided to use tools.)

In [ ]:
from claude_agent_sdk import query, ClaudeAgentOptions, AssistantMessage, TextBlock


async def ask_agent(prompt):
    options = ClaudeAgentOptions(model=SONNET, max_turns=1)  # single turn, for a fair comparison
    text = ""
    async for message in query(prompt=prompt, options=options):
        if isinstance(message, AssistantMessage):
            text += "".join(b.text for b in message.content if isinstance(b, TextBlock))
    return text


show("🤖 Claude Agent SDK", await ask_agent(task_prompt), "magenta")

### Which should you use?
Both answered the same question, but the Agent SDK needed more code because it is built for **multi-step, tool-using agents**.

- **Messages API** → *"I have one prompt and want one answer."* (Chapters 2–5)
- **Agent SDK** → *"Go through this folder of contracts, flag non-standard termination clauses, and write a summary file."* — Claude decides what to do next.

## 🏁 Cheat Sheet

| Concept | One-line takeaway |
|---|---|
| **Tokens** | The unit models read and you pay for (~4 characters each) |
| **Context window** | Everything in a request must fit; long docs need chunking/RAG |
| **Temperature** | Low = consistent, high = varied |
| **Non-determinism** | Same input can give different wording → test for *correctness*, not exact match |
| **Model choice** | Match capability / latency / cost to the task |
| **Adaptive thinking** | Let Claude decide how much to reason |
| **Zero / one / few-shot** | Add only as many examples as the task needs |
| **Sync · Stream · Async · Batch** | Now · progressively · concurrently · in the background |
| **Messages API vs. Agent SDK** | One answer vs. a multi-step, tool-using agent |